# CA2 Team Project: generate your team's data

Module COMP H2003, Artificial Intelligence, 2026 to 2027. Lecturer Maurice Martin, TU Dublin.

This notebook creates **your team's own** data set for the domain you chose. It comes from your team code and your domain, so it is different from every other team's, and your lecturer can recreate it exactly.

1. Type your team code and your domain in the cell under **Step 1**.
2. Choose **Run › Run All Cells**.
3. A file called `CA2_<domain>_<team code>.csv` appears in the same folder as this notebook.

**One person runs this and shares the CSV with the team**, or everyone runs it: the same code and domain always give the same file. **Do not change any other cell**, and don't edit the CSV by hand.

## Step 1: Your team code and domain

Your team code is on Brightspace, for example `Mon-02`. Your domain is one of `"delivery"` (food delivery), `"music"` (music streaming) or `"gaa"` (county GAA injuries). Keep the quotation marks. **Once you generate data, keep the same domain for the whole project.**

In [1]:
TEAM_ID = "Mon-02"      # <-- your team code, e.g. "Wed-07"
DOMAIN = "music"     # <-- "delivery", "music" or "gaa"

## Step 2: The generator

You don't need to read this cell. It builds 3,000 rows for your domain and adds some real-world mess.

In [2]:
import hashlib
import re
import numpy as np
import pandas as pd

DOMAINS = ["delivery", "music", "gaa"]
PROBLEM_TYPES = ["duplicates", "codes", "blanks", "impossible", "outliers", "labels"]
TEAM_RE = r"^(Mon|Wed|Thu)-(0[1-9]|1[0-9]|99)$"

COUNTIES = ["Antrim", "Armagh", "Carlow", "Cavan", "Clare", "Cork", "Derry", "Donegal",
            "Down", "Dublin", "Fermanagh", "Galway", "Kerry", "Kildare", "Kilkenny", "Laois",
            "Leitrim", "Limerick", "Longford", "Louth", "Mayo", "Meath", "Monaghan", "Offaly",
            "Roscommon", "Sligo", "Tipperary", "Tyrone", "Waterford", "Westmeath", "Wexford", "Wicklow"]


def check_team(team_id, domain):
    if not re.match(TEAM_RE, str(team_id)):
        raise ValueError("TEAM_ID must look like 'Mon-02', 'Wed-10' or 'Thu-07': use your team code exactly.")
    if domain not in DOMAINS:
        raise ValueError("DOMAIN must be one of 'delivery', 'music' or 'gaa'.")


def seed_from(team_id, domain, salt="data"):
    """Turn the team code and domain into a whole-number seed."""
    text = f"{team_id}|{domain}|{salt}|COMP-H2003-CA2"
    return int(hashlib.sha256(text.encode()).hexdigest(), 16) % (2**32)


# ---------------------------------------------------------------- domain 1: food delivery
def make_delivery(rng, n):
    zone = rng.choice(["City centre", "Inner suburbs", "Outer suburbs"], size=n, p=[0.30, 0.45, 0.25])
    codes = {"City centre": ["D01", "D02", "D07", "D08"],
             "Inner suburbs": ["D03", "D04", "D06", "D6W", "D09"],
             "Outer suburbs": ["D11", "D12", "D15", "D24"]}
    postcode = np.array([codes[z][rng.randint(len(codes[z]))] for z in zone])
    lo = np.select([zone == "City centre", zone == "Inner suburbs"], [0.4, 1.5], 3.5)
    hi = np.select([zone == "City centre", zone == "Inner suburbs"], [3.0, 5.0], 7.5)
    distance_km = (lo + rng.uniform(0, 1, n) * (hi - lo)).round(1)
    day = rng.choice(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], size=n,
                     p=[0.11, 0.10, 0.11, 0.13, 0.19, 0.20, 0.16])
    hour = rng.choice(np.arange(11, 24), size=n,
                      p=np.array([3, 5, 4, 3, 3, 4, 7, 11, 13, 12, 9, 6, 4]) / 84)
    raining = rng.uniform(0, 1, n) < 0.38
    rain_mm = np.where(raining, np.clip(rng.exponential(2.0, n), 0.1, 15), 0).round(1)
    p_car = np.select([zone == "City centre", zone == "Inner suburbs"], [0.10, 0.22], 0.45)
    u = rng.uniform(0, 1, n)
    vehicle = np.where(u < p_car, "car", np.where(u < p_car + (1 - p_car) * 0.5, "e-bike", "bike"))
    food_type = rng.choice(["pizza", "burgers", "asian", "other"], size=n, p=[0.30, 0.25, 0.25, 0.20])
    items = np.clip(rng.poisson(2.2, n) + 1, 1, 12)
    match_day = (rng.uniform(0, 1, n) < 0.08).astype(int)
    rider_months = np.clip(rng.gamma(1.6, 8, n), 0, 60).round().astype(int)

    base_prep = pd.Series(food_type).map({"pizza": 15, "burgers": 11, "asian": 17, "other": 13}).to_numpy()
    weekend_peak = np.isin(day, ["Fri", "Sat"]) & (hour >= 18) & (hour <= 21)
    prep_min = np.clip(base_prep + 0.9 * items + 4 * weekend_peak + rng.normal(0, 2.5, n), 4, 45).round().astype(int)

    speed = pd.Series(vehicle).map({"bike": 14.0, "e-bike": 19.0, "car": 24.0}).to_numpy()
    two_wheels = vehicle != "car"
    speed = speed * np.where(two_wheels, 1 - np.minimum(0.40, 0.06 * rain_mm), 1.0)
    rush = np.isin(day, ["Mon", "Tue", "Wed", "Thu", "Fri"]) & (hour >= 17) & (hour <= 19)
    speed = speed * np.where((vehicle == "car") & rush, 0.60, 1.0)
    speed = speed * np.where(match_day == 1, np.where(vehicle == "car", 0.50, 0.80), 1.0)
    travel = distance_km / speed * 60
    wait = (2 + 5 * weekend_peak + 0.8 * rain_mm + np.where(zone == "Outer suburbs", rng.uniform(2, 10, n), 0)
            + 6 * match_day + rng.exponential(2.0, n))
    experience = -0.15 * np.minimum(rider_months, 24)
    delivery_min = np.clip(prep_min + wait + travel + experience + rng.normal(0, 3.0, n), 8, 150).round(1)

    prep_est = base_prep + 0.6 * items
    promised_min = (np.ceil((prep_est + 3.6 * distance_km + 6) / 5) * 5).astype(int)
    late = (delivery_min > promised_min + 5).astype(int)
    return pd.DataFrame({
        "postcode": postcode, "zone": zone, "distance_km": distance_km, "day": day, "hour": hour,
        "rain_mm": rain_mm, "vehicle": vehicle, "food_type": food_type, "items": items,
        "match_day": match_day, "rider_months": rider_months, "prep_min": prep_min,
        "promised_min": promised_min, "delivery_min": delivery_min, "late": late})


# ---------------------------------------------------------------- domain 2: music streaming
def make_music(rng, n):
    age_band = rng.choice(["16-24", "25-34", "35-49", "50+"], size=n, p=[0.55, 0.25, 0.13, 0.07])
    lo = pd.Series(age_band).map({"16-24": 16, "25-34": 25, "35-49": 35, "50+": 50}).to_numpy()
    hi = pd.Series(age_band).map({"16-24": 24, "25-34": 34, "35-49": 49, "50+": 75}).to_numpy()
    listener_age = (lo + rng.uniform(0, 1, n) * (hi - lo + 1)).astype(int)
    genres = ["pop", "hip-hop", "rock", "dance", "indie", "trad", "country"]
    genre = rng.choice(genres, size=n, p=[0.27, 0.14, 0.13, 0.12, 0.14, 0.08, 0.12])
    g = pd.Series(genre)
    irish_artist = (rng.uniform(0, 1, n) < g.map({"pop": 0.12, "hip-hop": 0.10, "rock": 0.25, "dance": 0.10,
                                                    "indie": 0.40, "trad": 0.95, "country": 0.20}).to_numpy()).astype(int)
    track_length_sec = rng.uniform(120, 330, n).round().astype(int)
    t_lo = g.map({"pop": 95, "hip-hop": 80, "rock": 100, "dance": 118, "indie": 90, "trad": 90, "country": 75}).to_numpy()
    t_hi = g.map({"pop": 130, "hip-hop": 100, "rock": 160, "dance": 132, "indie": 140, "trad": 135, "country": 125}).to_numpy()
    tempo_bpm = (t_lo + rng.uniform(0, 1, n) * (t_hi - t_lo)).round().astype(int)
    e_mu = g.map({"pop": 0.65, "hip-hop": 0.62, "rock": 0.78, "dance": 0.85, "indie": 0.55, "trad": 0.45, "country": 0.50}).to_numpy()
    energy = np.clip(rng.normal(e_mu, 0.12, n), 0.02, 0.99).round(2)
    d_mu = g.map({"pop": 0.68, "hip-hop": 0.78, "rock": 0.48, "dance": 0.80, "indie": 0.52, "trad": 0.50, "country": 0.55}).to_numpy()
    danceability = np.clip(rng.normal(d_mu, 0.10, n), 0.05, 0.98).round(2)
    valence = np.clip(rng.normal(0.5, 0.2, n), 0.02, 0.98).round(2)
    track_age_years = np.clip(rng.exponential(5.0, n), 0, 50).round().astype(int)
    context = rng.choice(["playlist", "album", "radio", "search", "shuffle"], size=n, p=[0.35, 0.15, 0.15, 0.12, 0.23])
    liked_artist = (rng.uniform(0, 1, n) < 0.35).astype(int)
    session_position = np.clip(rng.geometric(0.08, n), 1, 60).astype(int)
    hour = rng.choice(np.arange(24), size=n, p=np.array([3, 2, 1, 1, 1, 1, 2, 4, 6, 5, 4, 4, 5, 5, 5, 5, 6, 7, 7, 7, 7, 6, 5, 4]) / 103)
    device = rng.choice(["phone", "desktop", "smart_speaker"], size=n, p=[0.70, 0.18, 0.12])

    young_g = np.isin(genre, ["pop", "hip-hop", "dance"])
    older_g = np.isin(genre, ["rock", "country", "trad"])
    young = listener_age < 30
    affinity = np.where(young & young_g, -1.0, 0) + np.where(young & older_g, 1.0, 0) \
        + np.where(~young & older_g, -1.0, 0) + np.where(~young & young_g, 0.9, 0)
    night = (hour >= 22) | (hour <= 5)
    morning = (hour >= 6) & (hour <= 10)
    mood = np.where(night, 3.0 * (energy - 0.5), 0) + np.where(morning, -2.5 * (energy - 0.5), 0)
    ctx = pd.Series(context).map({"playlist": 0.0, "album": -1.4, "radio": 0.7, "search": -2.6, "shuffle": 1.3}).to_numpy()
    dev = pd.Series(device).map({"phone": 0.0, "desktop": -0.3, "smart_speaker": -1.3}).to_numpy()
    score = (-0.75 + ctx - 1.8 * liked_artist + 0.045 * session_position + dev + affinity + mood
             + 0.006 * (track_length_sec - 225) - 0.3 * irish_artist - 0.03 * track_age_years * (~young)
             + rng.normal(0, 0.3, n))
    p = 1 / (1 + np.exp(-1.5 * score))
    skip = rng.uniform(0, 1, n) < p
    frac = np.clip(0.78 + 0.15 * liked_artist - 0.004 * session_position - 0.25 * (score > 0) * 0.5
                   - 0.0006 * (track_length_sec - 225) + rng.normal(0, 0.08, n), 0.15, 1.0)
    played = np.where(skip, 2 + rng.beta(1.3, 2.5, n) * 27, np.maximum(30, frac * track_length_sec))
    seconds_played = np.round(played).astype(int)
    seconds_played = np.where(skip, np.minimum(seconds_played, 29), np.maximum(seconds_played, 30))
    skipped = (seconds_played < 30).astype(int)
    return pd.DataFrame({
        "listener_age": listener_age, "age_band": age_band, "genre": genre, "irish_artist": irish_artist,
        "track_length_sec": track_length_sec, "tempo_bpm": tempo_bpm, "energy": energy,
        "danceability": danceability, "valence": valence, "track_age_years": track_age_years,
        "context": context, "liked_artist": liked_artist, "session_position": session_position,
        "hour": hour, "device": device, "seconds_played": seconds_played, "skipped": skipped})


# ---------------------------------------------------------------- domain 3: county GAA injuries
def make_gaa(rng, n, divisions):
    county = rng.choice(COUNTIES, size=n)
    division = pd.Series(county).map(divisions).to_numpy()
    code = rng.choice(["football", "hurling"], size=n, p=[0.6, 0.4])
    position = rng.choice(["goalkeeper", "defender", "midfielder", "forward"], size=n, p=[0.08, 0.36, 0.20, 0.36])
    age = np.clip(rng.normal(25.5, 3.8, n), 18, 36).round().astype(int)
    season_phase = rng.choice(["pre-season", "league", "championship"], size=n, p=[0.25, 0.40, 0.35])
    training_load = np.clip(rng.normal(2900 - 120 * (division - 1), 550, n), 1200, 4800).round(-1).astype(int)
    load_spike = np.clip(rng.lognormal(0.02, 0.22, n), 0.5, 2.2).round(2)
    matches_last_4wk = np.clip(rng.poisson(np.where(season_phase == "pre-season", 0.8, 2.6)), 0, 8).astype(int)
    sleep_hours = np.clip(rng.normal(7.3, 0.8, n), 4.5, 9.8).round(1)
    previous_injuries = np.clip(rng.poisson(1.1 + 0.06 * (age - 18)), 0, 9).astype(int)
    surface = rng.choice(["grass", "3G"], size=n, p=[0.7, 0.3])
    dual_player = (rng.uniform(0, 1, n) < 0.12).astype(int)
    travel_km = rng.uniform(10, 300, n).round().astype(int)

    pos = pd.Series(position).map({"goalkeeper": -0.8, "defender": 0.0, "midfielder": 0.35, "forward": 0.1}).to_numpy()
    phase = pd.Series(season_phase).map({"pre-season": 0.3, "league": 0.0, "championship": 0.2}).to_numpy()
    score = (-3.4 + 5.0 * np.maximum(0, load_spike - 1.15) + 2.5 * np.maximum(0, 0.85 - load_spike)
             + 0.55 * previous_injuries + 0.07 * (age - 25) + 0.9 * np.maximum(0, 7.2 - sleep_hours)
             + 0.22 * matches_last_4wk + 1.1 * dual_player + 0.5 * (surface == "3G") + 0.003 * travel_km
             + 0.2 * (division - 1) + 1.6 * pos + 1.5 * phase + 0.0003 * (training_load - 2800) + rng.normal(0, 0.3, n))
    p = 1 / (1 + np.exp(-1.35 * (score + 0.25)))
    injured_flag = rng.uniform(0, 1, n) < p
    sev = rng.normal(2.2 + 0.06 * (age - 25) + 0.18 * previous_injuries + 0.8 * np.maximum(0, load_spike - 1.15)
                     + 0.2 * (code == "hurling") + 0.15 * np.maximum(0, 7.2 - sleep_hours), 0.4, n)
    days_missed = np.where(injured_flag, np.clip(np.exp(sev), 1, 56).round(), 0).astype(int)
    injured = (days_missed > 0).astype(int)
    return pd.DataFrame({
        "county": county, "division": division, "code": code, "position": position, "age": age,
        "season_phase": season_phase, "training_load": training_load, "load_spike": load_spike,
        "matches_last_4wk": matches_last_4wk, "sleep_hours": sleep_hours, "previous_injuries": previous_injuries,
        "surface": surface, "dual_player": dual_player, "travel_km": travel_km,
        "days_missed": days_missed, "injured": injured})


ID_COL = {"delivery": ("order_id", "R"), "music": ("play_id", "P"), "gaa": ("player_id", "G")}
LABEL_COL = {"delivery": "food_type", "music": "genre", "gaa": "county"}


def divisions_for(team_id):
    """Each team's league divisions: 8 counties per division, shuffled by team (fictional)."""
    r = np.random.RandomState(seed_from(team_id, "gaa", "divisions"))
    order = r.permutation(len(COUNTIES))
    return {COUNTIES[i]: int(k // 8) + 1 for k, i in enumerate(order)}


def make_clean(domain, rng, n, team_id):
    if domain == "delivery":
        return make_delivery(rng, n)
    if domain == "music":
        return make_music(rng, n)
    return make_gaa(rng, n, divisions_for(team_id))


def _variant(rng, value, title_case):
    opts = [value.upper(), value + " ", " " + value] + ([value.lower()] if title_case else [value.title()])
    opts = [o for o in opts if o != value]
    return opts[rng.randint(len(opts))]


def make_dataset(team_id, domain, with_details=False):
    """Return the raw (messy) dataset for one team and domain."""
    check_team(team_id, domain)
    rng = np.random.RandomState(seed_from(team_id, domain))
    df = make_clean(domain, rng, 3000, team_id)
    id_col, prefix = ID_COL[domain]
    df.insert(0, id_col, [f"{prefix}{i:05d}" for i in range(1, len(df) + 1)])

    chosen = sorted(rng.choice(PROBLEM_TYPES, size=4, replace=False).tolist(), key=PROBLEM_TYPES.index)
    counts = {p: 0 for p in PROBLEM_TYPES}
    rows = rng.permutation(len(df))
    pos = 0

    def take(k):
        nonlocal pos
        out = rows[pos:pos + k]
        pos += k
        return out

    def two(col_a, val_a, col_b, val_b):
        k = rng.randint(25, 61); r = take(k); half = k // 2
        df[col_a] = df[col_a].astype(float) if not isinstance(val_a, str) else df[col_a]
        df[col_b] = df[col_b].astype(float) if not isinstance(val_b, str) else df[col_b]
        df.loc[r[:half], col_a] = val_a(half) if callable(val_a) else val_a
        df.loc[r[half:], col_b] = val_b(k - half) if callable(val_b) else val_b
        return k

    if domain == "delivery":
        spec = {"codes": ("rider_months", -1, "rain_mm", 999),
                "blanks": ("distance_km", np.nan, "prep_min", np.nan),
                "impossible": ("hour", lambda m: rng.randint(25, 31, m), "items", lambda m: rng.randint(40, 100, m)),
                "outliers": ("distance_km", lambda m: rng.uniform(70, 110, m).round(1))}
    elif domain == "music":
        spec = {"codes": ("session_position", -1, "tempo_bpm", 999),
                "blanks": ("energy", np.nan, "danceability", np.nan),
                "impossible": ("listener_age", lambda m: rng.randint(120, 200, m), "hour", lambda m: rng.randint(25, 31, m)),
                "outliers": ("track_length_sec", lambda m: rng.randint(1500, 2400, m))}
    else:
        spec = {"codes": ("sleep_hours", -1, "matches_last_4wk", 99),
                "blanks": ("training_load", np.nan, "sleep_hours", np.nan),
                "impossible": ("age", lambda m: rng.randint(80, 150, m), "previous_injuries", lambda m: rng.randint(40, 100, m)),
                "outliers": ("travel_km", lambda m: rng.randint(1600, 2600, m))}

    for prob in ["codes", "blanks", "impossible"]:
        if prob in chosen:
            counts[prob] = two(*spec[prob])
    if "outliers" in chosen:
        col, f = spec["outliers"]
        k = rng.randint(25, 61); r = take(k); counts["outliers"] = k
        df[col] = df[col].astype(float)
        df.loc[r, col] = f(k)
    if "labels" in chosen:
        col = LABEL_COL[domain]
        k = rng.randint(25, 61); r = take(k); counts["labels"] = k
        df.loc[r, col] = [_variant(rng, v, domain == "gaa") for v in df.loc[r, col]]
    if "duplicates" in chosen:
        k = rng.randint(25, 61); r = take(k); counts["duplicates"] = k
        df = pd.concat([df, df.loc[r]], ignore_index=True)

    df = df.iloc[rng.permutation(len(df))].reset_index(drop=True)
    if with_details:
        return df, {"problems": chosen, "counts": counts}
    return df


def make_hidden(team_id, domain, secret, n=800):
    """Clean, unseen rows for marking (grader only)."""
    rng = np.random.RandomState(seed_from(team_id, domain, "hidden|" + str(secret)))
    df = make_clean(domain, rng, n, team_id)
    id_col, prefix = ID_COL[domain]
    df.insert(0, id_col, [f"H{i:05d}" for i in range(1, len(df) + 1)])
    return df

## Step 3: Save your file

In [3]:
raw_df = make_dataset(TEAM_ID, DOMAIN)
data_file = f"CA2_{DOMAIN}_{TEAM_ID}.csv"
raw_df.to_csv(data_file, index=False)
print("Saved", data_file, "with", len(raw_df), "rows and", raw_df.shape[1], "columns")
print("Columns:", ", ".join(raw_df.columns))

Saved CA2_music_Mon-02.csv with 3000 rows and 18 columns
Columns: play_id, listener_age, age_band, genre, irish_artist, track_length_sec, tempo_bpm, energy, danceability, valence, track_age_years, context, liked_artist, session_position, hour, device, seconds_played, skipped


Now open the starter notebook for your domain: `CA2_starter_delivery.ipynb`, `CA2_starter_music.ipynb` or `CA2_starter_gaa.ipynb`.